# Job Recommendation System
## Step 3: Data Preprocessing

**Internship Project** | Gamage Recruiters  
**Focus:** Recruitment & HR / Data Science & Machine Learning

---

In this step we transform the raw datasets from Step 2 into clean, numeric feature representations that the recommendation engine can consume in Step 4.

| Task | What we do |
|---|---|
| **Text Cleaning** | Lowercase, remove punctuation, remove stopwords |
| **Skill Tokenization** | Convert comma-separated skill strings into TF-IDF tokens |
| **Combined Text** | Merge skills + summary/description into one feature field |
| **Categorical Encoding** | Ordinal-encode experience, label-encode education & domain |
| **Numeric Normalization** | Min-Max scale years of experience and salary |
| **TF-IDF Vectorization** | Build shared vocabulary matrix (300 features) across both datasets |

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer

print('Libraries loaded successfully')

Libraries loaded successfully


In [16]:
import os

# Create directories if they don't exist
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)

## 2. Load Raw Datasets

In [3]:
df_c = pd.read_csv('../data/candidates.csv')
df_j = pd.read_csv('../data/job_postings.csv')

print(f'Candidates  : {df_c.shape[0]} rows × {df_c.shape[1]} columns')
print(f'Job Postings: {df_j.shape[0]} rows × {df_j.shape[1]} columns')
print(f'\nCandidate columns : {list(df_c.columns)}')
print(f'Job columns       : {list(df_j.columns)}')

Candidates  : 500 rows × 15 columns
Job Postings: 200 rows × 15 columns

Candidate columns : ['candidate_id', 'name', 'email', 'phone', 'location', 'education_level', 'university', 'field_of_study', 'years_of_experience', 'experience_level', 'primary_domain', 'skills', 'preferred_locations', 'preferred_job_types', 'summary']
Job columns       : ['job_id', 'job_title', 'company_name', 'industry', 'location', 'job_type', 'experience_level', 'min_experience_years', 'education_required', 'required_skills', 'preferred_skills', 'salary_min_lkr', 'salary_max_lkr', 'posted_date', 'description']


In [4]:
# Quick check — missing values and data types
print('=== CANDIDATES — Missing Values ===')
print(df_c.isnull().sum())
print()
print('=== JOB POSTINGS — Missing Values ===')
print(df_j.isnull().sum())

=== CANDIDATES — Missing Values ===
candidate_id           0
name                   0
email                  0
phone                  0
location               0
education_level        0
university             0
field_of_study         0
years_of_experience    0
experience_level       0
primary_domain         0
skills                 0
preferred_locations    0
preferred_job_types    0
summary                0
dtype: int64

=== JOB POSTINGS — Missing Values ===
job_id                  0
job_title               0
company_name            0
industry                0
location                0
job_type                0
experience_level        0
min_experience_years    0
education_required      0
required_skills         0
preferred_skills        0
salary_min_lkr          0
salary_max_lkr          0
posted_date             0
description             0
dtype: int64


## 3. Define Stopwords

We use a custom stopword list instead of NLTK (no internet required). It covers common English stopwords **plus** recruitment-specific filler words that carry no matching signal (e.g., *'looking'*, *'opportunities'*, *'collaborate'*).

In [6]:
STOPWORDS = {
    # Common English
    'a','an','the','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had','do','does',
    'did','will','would','could','should','may','might','shall','can','need',
    'i','we','you','he','she','it','they','them','their','our','your','its',
    'this','that','these','those','from','by','as','about','into','through',
    'during','before','after','above','below','between','each','few','more',
    'most','other','some','such','no','not','only','same','so','than','too',
    'very','just','also','both','either',
    # Recruitment filler (no matching value)
    'looking','opportunities','level','professional','years','experience',
    'skilled','graduated','join','team','ideal','candidate','strong','skills',
    'work','exciting','projects','collaborate','cross','functional','teams',
    'preferred','include','none','related','field'
}

print(f'Stopword list defined — {len(STOPWORDS)} words')

Stopword list defined — 108 words


## 4. Text Cleaning Functions

In [7]:
def clean_text(text):
    """
    Clean a free-text field (summary / description):
    1. Lowercase
    2. Remove punctuation and special characters
    3. Remove stopwords and single-character tokens
    """
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)   # keep only letters, digits, spaces
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)


def clean_skills(skill_str):
    """
    Convert a comma-separated skill string into space-separated TF-IDF tokens.
    Multi-word skills (e.g. 'Machine Learning') are joined with underscore
    so TF-IDF treats them as a single token.
    """
    if pd.isna(skill_str):
        return ''
    skills = [s.strip().lower() for s in str(skill_str).split(',')]
    skills = [re.sub(r'\s+', '_', s) for s in skills if s]   # 'machine learning' → 'machine_learning'
    return ' '.join(skills)


# ── Demo ──────────────────────────────────────────────────────────────────
sample_summary = df_c['summary'].iloc[0]
sample_skills  = df_c['skills'].iloc[0]

print('─── clean_text demo ───')
print(f'Original : {sample_summary}')
print(f'Cleaned  : {clean_text(sample_summary)}')
print()
print('─── clean_skills demo ───')
print(f'Original : {sample_skills}')
print(f'Cleaned  : {clean_skills(sample_skills)}')

─── clean_text demo ───
Original : Junior-level professional with 3 years of experience in Technology. Skilled in MongoDB, scikit-learn, PyTorch. Graduated from University of Jaffna. Looking for Freelance opportunities in Polonnaruwa.
Cleaned  : junior technology mongodb scikit learn pytorch university jaffna freelance polonnaruwa

─── clean_skills demo ───
Original : MongoDB, scikit-learn, PyTorch, TensorFlow
Cleaned  : mongodb scikit-learn pytorch tensorflow


## 5. Preprocess Candidate Data

### 5a. Text Fields

In [8]:
# Clean text columns
df_c['skills_clean']  = df_c['skills'].apply(clean_skills)
df_c['summary_clean'] = df_c['summary'].apply(clean_text)

# Combined text — skills weighted higher by appearing first
# Skills are repeated once to give them more TF-IDF weight vs. freetext summary
df_c['combined_text'] = df_c['skills_clean'] + ' ' + df_c['skills_clean'] + ' ' + df_c['summary_clean']

print('Candidate text fields cleaned')
print(f'\nSample skills_clean  : {df_c["skills_clean"].iloc[0]}')
print(f'Sample summary_clean : {df_c["summary_clean"].iloc[0]}')
print(f'Sample combined_text : {df_c["combined_text"].iloc[0][:120]}...')

Candidate text fields cleaned

Sample skills_clean  : mongodb scikit-learn pytorch tensorflow
Sample summary_clean : junior technology mongodb scikit learn pytorch university jaffna freelance polonnaruwa
Sample combined_text : mongodb scikit-learn pytorch tensorflow mongodb scikit-learn pytorch tensorflow junior technology mongodb scikit learn p...


### 5b. Categorical Encoding

In [9]:
# Ordinal encoding — experience level has a natural order
exp_order = {'Entry': 0, 'Junior': 1, 'Mid': 2, 'Senior': 3, 'Lead': 4, 'Manager': 5}
df_c['experience_encoded'] = df_c['experience_level'].map(exp_order)

# Label encoding — education and domain (no natural order)
le_edu = LabelEncoder()
le_dom = LabelEncoder()
df_c['education_encoded'] = le_edu.fit_transform(df_c['education_level'])
df_c['domain_encoded']    = le_dom.fit_transform(df_c['primary_domain'])

print('Categorical columns encoded')
print(f'\nExperience mapping : {exp_order}')
print(f'Education classes  : {list(le_edu.classes_)}')
print(f'Domain classes     : {list(le_dom.classes_)}')
print()
print(df_c[['experience_level','experience_encoded','education_level',
            'education_encoded','primary_domain','domain_encoded']].drop_duplicates().sort_values('experience_encoded').head(10))

Categorical columns encoded

Experience mapping : {'Entry': 0, 'Junior': 1, 'Mid': 2, 'Senior': 3, 'Lead': 4, 'Manager': 5}
Education classes  : ["Bachelor's", 'Diploma', 'High School', "Master's", 'PhD']
Domain classes     : ['Finance', 'HR', 'Marketing', 'Operations', 'Technology']

   experience_level  experience_encoded education_level  education_encoded  \
3             Entry                   0      Bachelor's                  0   
7             Entry                   0             PhD                  4   
6             Entry                   0         Diploma                  1   
11            Entry                   0         Diploma                  1   
33            Entry                   0        Master's                  3   
24            Entry                   0             PhD                  4   
21            Entry                   0        Master's                  3   
19            Entry                   0        Master's                  3   
55          

### 5c. Numeric Normalization

In [10]:
scaler_c = MinMaxScaler()
df_c['experience_normalized'] = scaler_c.fit_transform(df_c[['years_of_experience']])

print('years_of_experience normalized to [0, 1]')
print(f'\nOriginal range : {df_c["years_of_experience"].min()} – {df_c["years_of_experience"].max()} years')
print(f'Scaled range   : {df_c["experience_normalized"].min():.2f} – {df_c["experience_normalized"].max():.2f}')
print()
df_c[['years_of_experience','experience_normalized']].describe().round(3)

years_of_experience normalized to [0, 1]

Original range : 0 – 12 years
Scaled range   : 0.00 – 1.00



,years_of_experience,experience_normalized
count,500.000,500.000
mean,5.882,0.490
std,3.894,0.324
min,0.000,0.000
25%,2.000,0.167
50%,6.000,0.500
75%,10.000,0.833
max,12.000,1.000


## 6. Preprocess Job Postings Data

### 6a. Text Fields

In [11]:
df_j['required_skills_clean']  = df_j['required_skills'].apply(clean_skills)
df_j['preferred_skills_clean'] = df_j['preferred_skills'].apply(clean_skills)
df_j['description_clean']      = df_j['description'].apply(clean_text)

# Required skills are most important — repeat twice for extra TF-IDF weight
df_j['combined_text'] = (df_j['required_skills_clean'] + ' ' +
                          df_j['required_skills_clean'] + ' ' +
                          df_j['preferred_skills_clean'] + ' ' +
                          df_j['description_clean'])

print('Job posting text fields cleaned')
print(f'\nSample required_skills_clean : {df_j["required_skills_clean"].iloc[0]}')
print(f'Sample description_clean     : {df_j["description_clean"].iloc[0][:120]}...')

Job posting text fields cleaned

Sample required_skills_clean : teamwork financial_modeling tax_planning tableau accounting python fixed_income presentation_skills power_bi
Sample description_clean     : mid financial analyst finance pan asia bank teamwork financial modeling tax planning gaap equity research...


### 6b. Categorical Encoding

In [12]:
# Reuse the SAME encoders fitted on candidate data — shared label space
df_j['experience_encoded']   = df_j['experience_level'].map(exp_order)
df_j['edu_required_encoded'] = le_edu.transform(
    df_j['education_required'].apply(
        lambda x: x if x in le_edu.classes_ else "Bachelor's"  # fallback
    )
)
df_j['industry_encoded'] = le_dom.transform(
    df_j['industry'].apply(
        lambda x: x if x in le_dom.classes_ else 'Technology'  # fallback
    )
)

print('Job posting categorical columns encoded (same encoders as candidates)')
print()
print(df_j[['experience_level','experience_encoded','industry','industry_encoded']]
      .drop_duplicates().sort_values('experience_encoded').head(10))

Job posting categorical columns encoded (same encoders as candidates)

   experience_level  experience_encoded    industry  industry_encoded
7             Entry                   0  Technology                 4
4             Entry                   0          HR                 1
9             Entry                   0   Marketing                 2
29            Entry                   0  Operations                 3
46            Entry                   0     Finance                 0
2            Junior                   1  Operations                 3
23           Junior                   1     Finance                 0
1            Junior                   1  Technology                 4
61           Junior                   1          HR                 1
69           Junior                   1   Marketing                 2


### 6c. Numeric Normalization

In [13]:
# Normalize minimum experience years
scaler_j = MinMaxScaler()
df_j['min_exp_normalized'] = scaler_j.fit_transform(df_j[['min_experience_years']])

# Normalize salary range and compute midpoint
sal_scaler = MinMaxScaler()
df_j[['salary_min_norm', 'salary_max_norm']] = sal_scaler.fit_transform(
    df_j[['salary_min_lkr', 'salary_max_lkr']]
)
df_j['salary_mid_norm'] = (df_j['salary_min_norm'] + df_j['salary_max_norm']) / 2

print('Numeric fields normalized')
print(f'\nSalary (LKR) range  : {df_j["salary_min_lkr"].min():,} – {df_j["salary_max_lkr"].max():,}')
print(f'Salary norm range   : {df_j["salary_min_norm"].min():.2f} – {df_j["salary_max_norm"].max():.2f}')
print()
df_j[['salary_min_lkr','salary_max_lkr','salary_min_norm','salary_max_norm','salary_mid_norm']].describe().round(3)

Numeric fields normalized

Salary (LKR) range  : 35,629 – 548,666
Salary norm range   : 0.00 – 1.00



,salary_min_lkr,salary_max_lkr,salary_min_norm,salary_max_norm,salary_mid_norm
count,200.000,200.000,200.000,200.000,200.000
mean,170768.405,225908.550,0.348,0.356,0.352
std,118006.678,155363.156,0.304,0.310,0.305
min,35629.000,47671.000,0.000,0.000,0.002
25%,61475.250,86459.750,0.067,0.077,0.077
50%,129720.000,168665.000,0.243,0.242,0.232
75%,277740.750,365718.250,0.624,0.635,0.633
max,423588.000,548666.000,1.000,1.000,0.955


## 7. TF-IDF Vectorization

We fit a **single shared TF-IDF vocabulary** on combined candidate + job texts. This ensures both matrices live in the same feature space, which is required for cosine similarity computation in Step 4.

- `max_features=300` — keeps the 300 most informative tokens
- `ngram_range=(1,2)` — captures single words AND bi-grams (e.g., `machine_learning`)
- `min_df=2` — ignores tokens that appear in fewer than 2 documents (noise reduction)

In [14]:
# Fit on all texts (candidates + jobs) to build a shared vocabulary
all_texts = pd.concat([df_c['combined_text'], df_j['combined_text']], ignore_index=True)

tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True   # apply log normalization to term frequency
)
tfidf.fit(all_texts)

# Transform both datasets
candidate_tfidf = tfidf.transform(df_c['combined_text'])
job_tfidf       = tfidf.transform(df_j['combined_text'])

print('TF-IDF vectorization complete')
print(f'\nVocabulary size         : {len(tfidf.vocabulary_)} tokens')
print(f'Candidate matrix shape  : {candidate_tfidf.shape}  (500 candidates × 300 features)')
print(f'Job matrix shape        : {job_tfidf.shape}  (200 jobs × 300 features)')
print(f'\nSample vocabulary tokens: {list(tfidf.vocabulary_.keys())[:20]}')

TF-IDF vectorization complete

Vocabulary size         : 300 tokens
Candidate matrix shape  : (500, 300)  (500 candidates × 300 features)
Job matrix shape        : (200, 300)  (200 jobs × 300 features)

Sample vocabulary tokens: ['mongodb', 'scikit', 'learn', 'pytorch', 'tensorflow', 'junior', 'technology', 'university', 'jaffna', 'freelance', 'polonnaruwa', 'scikit learn', 'tax_planning', 'attention_to_detail', 'financial_modeling', 'portfolio_management', 'tableau', 'equity_research', 'gaap', 'analytical_thinking']


### Top TF-IDF Terms

In [15]:
# Show the most common TF-IDF terms in the vocabulary
feature_names = tfidf.get_feature_names_out()
candidate_means = np.asarray(candidate_tfidf.mean(axis=0)).flatten()
job_means       = np.asarray(job_tfidf.mean(axis=0)).flatten()

top_candidate_terms = pd.Series(candidate_means, index=feature_names).sort_values(ascending=False).head(20)
top_job_terms       = pd.Series(job_means,       index=feature_names).sort_values(ascending=False).head(20)

print('Top 20 TF-IDF terms — Candidates:')
print(top_candidate_terms.to_string())
print()
print('Top 20 TF-IDF terms — Job Postings:')
print(top_job_terms.to_string())

Top 20 TF-IDF terms — Candidates:
university            0.055557
communication         0.054634
excel                 0.053794
sql                   0.049112
data_analysis         0.047980
leadership            0.047961
teamwork              0.043854
time                  0.042531
reporting             0.035033
marketing             0.033613
project_management    0.032369
technology            0.031189
internship            0.028954
full time             0.028944
full                  0.028667
management            0.027976
sri                   0.027818
contract              0.027791
hr                    0.027679
part time             0.026719

Top 20 TF-IDF terms — Job Postings:
communication         0.064427
leadership            0.057974
excel                 0.057716
data_analysis         0.057352
teamwork              0.056215
analyst               0.054777
reporting             0.052627
sql                   0.049745
management            0.047994
marketing             0.047861

## 8. Save Preprocessed Datasets

In [17]:
import scipy.sparse as sp
import pickle

# Save enriched CSVs (original columns + all new preprocessed columns)
df_c.to_csv('../data/candidates_preprocessed.csv', index=False)
df_j.to_csv('../data/job_postings_preprocessed.csv', index=False)

# Save TF-IDF matrices (sparse format — efficient for large datasets)
sp.save_npz('../models/candidate_tfidf_matrix.npz', candidate_tfidf)
sp.save_npz('../models/job_tfidf_matrix.npz', job_tfidf)

# Save TF-IDF vectorizer and encoders for reuse in Step 4
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('../models/encoders.pkl', 'wb') as f:
    pickle.dump({'le_edu': le_edu, 'le_dom': le_dom, 'exp_order': exp_order,
                 'scaler_c': scaler_c, 'scaler_j': scaler_j, 'sal_scaler': sal_scaler}, f)

print('All outputs saved:')
print('   ../data/candidates_preprocessed.csv')
print('   ../data/job_postings_preprocessed.csv')
print('   ../models/candidate_tfidf_matrix.npz')
print('   ../models/job_tfidf_matrix.npz')
print('   ../models/tfidf_vectorizer.pkl')
print('   ../models/encoders.pkl')

All outputs saved:
   ../data/candidates_preprocessed.csv
   ../data/job_postings_preprocessed.csv
   ../models/candidate_tfidf_matrix.npz
   ../models/job_tfidf_matrix.npz
   ../models/tfidf_vectorizer.pkl
   ../models/encoders.pkl


## 9. Final Verification

In [18]:
print('=== NEW COLUMNS ADDED — CANDIDATES ===')
new_c_cols = ['skills_clean','summary_clean','combined_text',
              'experience_encoded','education_encoded','domain_encoded','experience_normalized']
print(df_c[new_c_cols].head(3).to_string())
print()
print('=== NEW COLUMNS ADDED — JOB POSTINGS ===')
new_j_cols = ['required_skills_clean','description_clean','combined_text',
              'experience_encoded','industry_encoded','min_exp_normalized',
              'salary_min_norm','salary_max_norm','salary_mid_norm']
print(df_j[new_j_cols].head(3).to_string())

=== NEW COLUMNS ADDED — CANDIDATES ===
                                                                                                                                           skills_clean                                                                                   summary_clean                                                                                                                                                                                                                                                                                                                                                                              combined_text  experience_encoded  education_encoded  domain_encoded  experience_normalized
0                                                                                                               mongodb scikit-learn pytorch tensorflow          junior technology mongodb scikit learn pytorch university jaffna freelance polonnaruwa         

In [19]:
print('=== PREPROCESSING SUMMARY ===')
print(f'\nCandidates shape (preprocessed) : {df_c.shape}')
print(f'Job Postings shape (preprocessed): {df_j.shape}')
print(f'Candidate TF-IDF matrix         : {candidate_tfidf.shape}')
print(f'Job TF-IDF matrix               : {job_tfidf.shape}')
print(f'Shared vocabulary size          : {len(tfidf.vocabulary_)}')
print(f'Missing values — candidates     : {df_c.isnull().sum().sum()}')
print(f'Missing values — job postings   : {df_j.isnull().sum().sum()}')
print()
print('Encoded columns:')
print('  experience_level  -> experience_encoded  (ordinal: 0–5)')
print('  education_level   -> education_encoded   (label encoded)')
print('  primary_domain    -> domain_encoded      (label encoded)')
print('  years_of_experience -> experience_normalized (MinMax [0,1])')
print('  salary_min/max_lkr  -> salary_min/max_norm  (MinMax [0,1])')

=== PREPROCESSING SUMMARY ===

Candidates shape (preprocessed) : (500, 22)
Job Postings shape (preprocessed): (200, 26)
Candidate TF-IDF matrix         : (500, 300)
Job TF-IDF matrix               : (200, 300)
Shared vocabulary size          : 300
Missing values — candidates     : 0
Missing values — job postings   : 0

Encoded columns:
  experience_level  -> experience_encoded  (ordinal: 0–5)
  education_level   -> education_encoded   (label encoded)
  primary_domain    -> domain_encoded      (label encoded)
  years_of_experience -> experience_normalized (MinMax [0,1])
  salary_min/max_lkr  -> salary_min/max_norm  (MinMax [0,1])


## 10. Summary

| Output File | Description |
|---|---|
| `candidates_preprocessed.csv` | Original candidate data + all cleaned & encoded columns |
| `job_postings_preprocessed.csv` | Original job data + all cleaned & encoded columns |
| `candidate_tfidf_matrix.npz` | Sparse TF-IDF matrix — 500 × 300 |
| `job_tfidf_matrix.npz` | Sparse TF-IDF matrix — 200 × 300 |
| `tfidf_vectorizer.pkl` | Fitted TF-IDF vectorizer (reuse in Step 4+) |
| `encoders.pkl` | All fitted encoders & scalers (reuse in Step 4+) |

### Design Decisions
- **Skills repeated in combined_text** — gives them higher TF-IDF weight vs. generic summary words
- **Shared TF-IDF vocabulary** — candidates and jobs are in the same feature space, enabling direct cosine similarity
- **Multi-word skills underscored** — `machine_learning` is one token, not two separate weak signals
- **Ordinal encoding for experience** — preserves the natural ordering (Entry < Junior < Mid < Senior)
- **Same encoders on both datasets** — ensures label consistency (e.g., Finance=1 means Finance in both)

### Next Step: Step 4 — Recommendation Engine (Content-Based Filtering)
- Load `candidate_tfidf_matrix.npz` and `job_tfidf_matrix.npz`
- Compute cosine similarity between each candidate and all job postings
- Rank and return Top-N job recommendations per candidate